# 01 — Kiểm chứng bộ lọc trên FPGA PYNQ-Z2

Notebook này tự đọc cấu hình board, nạp overlay và so **mọi pixel** với kết quả chuẩn. Không cần chạy notebook inventory riêng trước đó, không cần màn hình HDMI ngoài.

1. Upload **`START_HERE.ipynb`** và **`video_pynq_z2_deploy.zip`** vào cùng thư mục Jupyter **của board**. Mở `START_HERE.ipynb`, chọn **Run All** để kiểm tra SHA256 và giải nén thành thư mục `video_demo`; không cần Terminal.
2. Dừng chương trình MNIST hoặc overlay khác. Mở notebook này trong `video_demo`, chọn **Kernel → Restart & Run All**.
3. Chờ báo **PHYSICAL FPGA PASS**. Báo cáo thật được lưu `results/board_dma_results.json`.

Nếu có lỗi, notebook dừng và lưu FAIL, không thay kết quả FPGA bằng phép tính Python. Bitstream đã build/mô phỏng trên PC chỉ được xác nhận chạy board sau khi notebook này PASS.


In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / 'video_driver.py').is_file():
    raise RuntimeError('Mở notebook trong thư mục video_demo đã giải nén trên board, có video_driver.py cạnh notebook.')
sys.path.insert(0, str(ROOT))
RESULTS = ROOT / 'results'
RESULTS.mkdir(exist_ok=True)
print('Thư mục demo:', ROOT)


## Đọc cấu hình tự động
Cell này không hỏi thêm thông tin và không nạp FPGA. Nó ghi cấu hình trong báo cáo để hỗ trợ khi image PYNQ khác phiên bản.


In [ ]:
from board_inventory import collect_inventory
inventory = collect_inventory()
(RESULTS / 'board_inventory.json').write_text(json.dumps(inventory, indent=2, ensure_ascii=False), encoding='utf-8')
print('Board:', inventory['board_model'] or inventory['board_env'])
print('Python:', inventory['python'], '| PYNQ:', inventory['packages']['pynq'])
print('PYNQ package version có thể trống nếu image cũ không có distribution metadata.')


## Nạp overlay và kiểm thử đầy đủ

- Kiểm tra SHA256 `.bit/.hwh`, ID/version và FCLK0 100 MHz.
- 23 ảnh × 4 mode = **92 phép đối chiếu toàn bộ pixel**, gồm VGA 640×480.
- **100 frame liên tiếp**, thay mode/kích thước và reset ba lần.
- Mỗi mode benchmark riêng **100 frame VGA**, sau 5 frame warm-up. Không hiển thị ảnh trong khoảng đo.

Chu kỳ core gồm stall/drain; thời gian cả driver gồm đóng gói, cache, DMA, MMIO và đọc lại. FPS trình duyệt và HDMI được đo riêng ở demo tương ứng.


In [ ]:
from board_workflows import run_dma_verification
report = run_dma_verification(ROOT, timeout=5.0)
print('Kết quả:', report['status'])
print('Đối chiếu pixel:', len(report['pixel_tests']), '| Frame liên tiếp:', report['sequential_frames'])


## Xem số đo trên board
Số dưới đây là số đo của lần chạy hiện tại trên ARM/FPGA. Không dùng số dự đoán từ simulation làm tốc độ board.


In [ ]:
from IPython.display import display, HTML
rows=[]
for b in report['benchmarks']:
    rows.append('<tr><td>%d</td><td>%.3f</td><td>%.3f</td><td>%.3f</td><td>%.1f</td><td>%.3f</td></tr>' % (
        b['mode'],b['core_active']['mean_ms'],b['dma_and_control']['mean_ms'],
        b['whole_driver']['mean_ms'],b['whole_driver']['frames_per_second_from_mean'],
        b['cpu_numpy_reference']['mean_ms']))
display(HTML('<table><tr><th>Mode</th><th>Core ms</th><th>DMA+control ms</th><th>Driver ms</th><th>Driver frame/s</th><th>ARM NumPy ms</th></tr>'+''.join(rows)+'</table>'))
print('Snapshot FPGA: results/fpga_mode0.png ... fpga_mode3.png')
print('Tải results/board_dma_results.json về laptop để lưu bằng chứng.')


## Tiếp theo
Mở `02_dma_preview.ipynb` để xem ảnh và điều khiển bốn bộ lọc. Bộ đệm DMA của notebook kiểm thử đã được giải phóng. Bản A chạy trên FPGA qua DMA; đây chưa phải demo HDMI vào/ra liên tục.
